In [ ]:
import sys
print(sys.executable)

!which pip
!which python
!pip3 show torchvision

In [ ]:
!python3 --version

In [ ]:
#load the MNIST dataset and get 50 principal components

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
import numpy as np
import matplotlib.pyplot as plt

def get_mnist_data(batch_size=1024):
    """
    Loads MNIST using PyTorch DataLoader with standard normalization.
    Returns the full dataset as numpy arrays (N, 784) and labels.
    """
    print("Initializing PyTorch DataLoader...")
    
    # 1. Define Transforms
    # ToTensor converts [0, 255] -> [0.0, 1.0]
    # Normalize subtracts mean (0.1307) and divides by std (0.3081)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    # 2. Load Dataset
    train_dataset = torchvision.datasets.MNIST(
        root='./data', 
        train=True, 
        transform=transform, 
        download=True
    )
    
    # 3. Create DataLoader
    # We use a large batch size here since we ultimately want the whole dataset
    loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=False)

    all_data = []
    all_labels = []

    # 4. Extract data from Loader
    for images, labels in loader:
        # Flatten images: [Batch, 1, 28, 28] -> [Batch, 784]
        flat_images = images.view(images.size(0), -1).numpy()
        all_data.append(flat_images)
        all_labels.append(labels.numpy())

    # Concatenate all batches into single numpy arrays
    X = np.vstack(all_data)
    y = np.concatenate(all_labels)
    
    print(f"Data Loaded. Shape: {X.shape}")
    return X, y

def apply_pca(X, n_components=50):
    """
    Applies PCA to reduce dimensionality to n_components.
    """
    print(f"Running PCA to reduce dimensions from {X.shape[1]} to {n_components}...")
    
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X)
    
    explained_variance = np.sum(pca.explained_variance_ratio_)
    print(f"PCA Complete. New Shape: {X_pca.shape}")
    print(f"Total Explained Variance: {explained_variance:.2%}")
    
    return X_pca, pca



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import kneighbors_graph
from ripser import ripser
from scipy.stats import mode
from tqdm import tqdm  # Progress bar
import warnings

warnings.filterwarnings("ignore")

def check_homology_fit(diagrams, radii_range):
    """
    Evaluates topological fit across the specific radius range:
    1.0, 1.5, 2.0, ..., 5.0
    
    Returns:
    - connected_at: The radius where B0 becomes 1 (or None if never)
    - has_holes: Boolean, true if B1 exists (sum > 0)
    """
    dgm0 = diagrams[0] # Components
    dgm1 = diagrams[1] # Loops
    
    connected_at = None
    has_holes = False
    
    # Check if we have any loops (H1) that persist
    # We sum the lifetimes of H1 features
    if dgm1.shape[0] > 0:
        lifetimes = dgm1[:, 1] - dgm1[:, 0]
        # If we have significant loops
        if np.sum(lifetimes) > 0.1: 
            has_holes = True

    # Check connectivity across the user's specific range
    for r in radii_range:
        # Count components (B0) that are still 'alive' (death > r)
        b0_count = np.sum(dgm0[:, 1] > r)
        
        if b0_count == 1:
            connected_at = r
            break # Found the connection point
            
    return connected_at, has_holes

def monte_carlo_knn_topology(X_pool, n_trials=20, subsample_size=3000):
    # User definitions
    k_range = range(5, 31) 
    radii_check = np.arange(1, 5.5, 0.5) # [1.0, 1.5, ... 5.0]
    
    best_k_per_trial = []
    
    print(f"--- Starting Monte Carlo TDA ---")
    print(f"Trials: {n_trials} | Subsample Size: {subsample_size}")
    print(f"Checking Radii: {radii_check}")
    
    # Outer loop: Monte Carlo Trials
    for trial in tqdm(range(n_trials), desc="Monte Carlo Trials"):
        
        # 1. Subsample
        indices = np.random.choice(X_pool.shape[0], subsample_size, replace=False)
        X_sub = X_pool[indices]
        
        # We will store candidate k values that worked for this specific subsample
        # Format: (k_value, connection_radius)
        valid_candidates = []

        # 2. Iterate through k
        for k in k_range:
            # Build Graph
            knn_matrix = kneighbors_graph(X_sub, k, mode='distance', include_self=False)
            
            # Compute Persistence (Up to H1)
            result = ripser(knn_matrix, maxdim=1, distance_matrix=True)
            
            # 3. Analyze across Radii Range
            conn_radius, valid_h1 = check_homology_fit(result['dgms'], radii_check)
            
            # CRITERIA:
            # 1. Must connect (B0=1) at some point within [1, 5]
            # 2. Must not be "overconnected" (implied if it connects instantly at r=1)
            # 3. Ideally preserves H1 features (digits have loops)
            if conn_radius is not None:
                valid_candidates.append({
                    'k': k,
                    'r_connect': conn_radius,
                    'has_holes': valid_h1
                })

        # 4. Pick best k for this trial
        if valid_candidates:
            # Sort candidates.
            # Priority 1: Has Holes (We want to capture digit structure)
            # Priority 2: Smallest k (simplest graph that works)
            
            # Filter for candidates that preserved holes
            candidates_with_holes = [c for c in valid_candidates if c['has_holes']]
            
            if candidates_with_holes:
                # Pick smallest k that preserved holes and connected
                best_k = min(candidates_with_holes, key=lambda x: x['k'])['k']
            else:
                # If no holes found, just pick smallest k that connected
                best_k = min(valid_candidates, key=lambda x: x['k'])['k']
                
            best_k_per_trial.append(best_k)
        
    # --- FINAL RESULT ---
    if best_k_per_trial:
        # Calculate Mode
        global_k_mode = mode(best_k_per_trial, keepdims=True).mode[0]
        
        # Visualization of the distribution of Best K
        plt.figure(figsize=(8, 4))
        plt.hist(best_k_per_trial, bins=range(5, 32), align='left', rwidth=0.8, color='skyblue', edgecolor='black')
        plt.axvline(global_k_mode, color='red', linestyle='dashed', linewidth=2, label=f'Mode: {global_k_mode}')
        plt.title(f"Distribution of Optimal k over {n_trials} Trials")
        plt.xlabel("k value")
        plt.ylabel("Frequency")
        plt.legend()
        plt.show()
        
        print("\n" + "="*40)
        print(f"Optimal Global k (Mode): {global_k_mode}")
        print("="*40)
        return global_k_mode
    else:
        print("Simulation failed. No k values satisfied constraints.")
        return None


In [ ]:
# --- Main Execution ---
if __name__ == "__main__":
    # 1. Load and Normalize
    X_raw, y = get_mnist_data()

    # 2. Run PCA (Reduce to 50 latent features)
    X_pca, pca_model = apply_pca(X_raw, n_components=50)

    # 3. Save or Prepare for Monte Carlo
    # This X_pca is now 'ready' for your KNN/VR complex simulation.
    # It is significantly smaller (50 vs 784 dims) and cleaner.
    
    print("\n--- Ready for Monte Carlo Simulation ---")
    print(f"Dataset X_pca size: {X_pca.nbytes / 1e6:.2f} MB")
    print("You can now subsample this matrix for your Vietoris-Rips analysis.")

    # Optional: Visual check of the variance captured
    plt.figure(figsize=(8, 5))
    plt.plot(np.cumsum(pca_model.explained_variance_ratio_))
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.title('PCA Analysis of MNIST')
    plt.grid(True)
    plt.show()

    best_k = monte_carlo_knn_topology(X_pca, n_trials=10) 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import kneighbors_graph
from ripser import ripser
from tqdm import tqdm
import scipy.sparse

def get_eta_candidates(X_pool, fixed_k, n_trials=10, subsample_size=3000):
    """
    Method 1: Percentile-based Etas
    Runs Monte Carlo to find robust edge length percentiles across subsamples.
    """
    print(f"--- Calculating Edge Length Percentiles (k={fixed_k}) ---")
    
    p25_log, p50_log, p75_log = [], [], []

    for _ in tqdm(range(n_trials), desc="Sampling Edge Distributions"):
        # 1. Subsample
        indices = np.random.choice(X_pool.shape[0], subsample_size, replace=False)
        X_sub = X_pool[indices]

        # 2. Build k-NN Graph
        # mode='distance' gives us the lengths (weights) of the edges
        knn_matrix = kneighbors_graph(X_sub, fixed_k, mode='distance', include_self=False)
        
        # 3. Extract Edge Lengths
        # The sparse matrix stores only existing edges. valid_distances represents 
        # the "lengths" of connections in the topology.
        valid_distances = knn_matrix.data 
        
        # 4. Compute Percentiles
        p25_log.append(np.percentile(valid_distances, 25))
        p50_log.append(np.percentile(valid_distances, 50))
        p75_log.append(np.percentile(valid_distances, 75))

    # Average over trials
    eta_25 = np.mean(p25_log)
    eta_50 = np.mean(p50_log)
    eta_75 = np.mean(p75_log)

    print(f"\nRecommended Eta values (Edge Length Percentiles):")
    print(f"  Eta (25th): {eta_25:.4f} (Conservative/Tight)")
    print(f"  Eta (50th): {eta_50:.4f} (Median Connectivity)")
    print(f"  Eta (75th): {eta_75:.4f} (Loose/Broad)")
    
    return eta_25, eta_50, eta_75

def visualize_plateau(X_pool, fixed_k, subsample_size=3000):
    """
    Method 2: The Plateau (Visual Inspection)
    Plots Betti curves to find the region where B0=1 and B1 persists.
    """
    print(f"\n--- Generating Plateau Visualization (Single Trial) ---")
    
    # 1. Subsample & Graph
    indices = np.random.choice(X_pool.shape[0], subsample_size, replace=False)
    X_sub = X_pool[indices]
    knn_matrix = kneighbors_graph(X_sub, fixed_k, mode='distance', include_self=False)
    
    # 2. Compute Persistence (Dense simulation for smooth curves)
    # We increase maxdim to 1 to see holes
    result = ripser(knn_matrix, maxdim=1, distance_matrix=True)
    dgms = result['dgms']
    
    # 3. Construct Betti Curves
    # We scan a fine range of radii to draw the lines
    radii = np.linspace(0, 5.0, 100)
    b0_vals = []
    b1_vals = []
    
    # dgms[0] = H0 intervals, dgms[1] = H1 intervals
    for r in radii:
        # B0: Count components alive at r
        b0 = np.sum(dgms[0][:, 1] > r)
        b0_vals.append(b0)
        
        # B1: Count loops alive at r
        if dgms[1].shape[0] > 0:
            b1 = np.sum((dgms[1][:, 0] <= r) & (dgms[1][:, 1] > r))
        else:
            b1 = 0
        b1_vals.append(b1)

    # 4. Plot
    plt.figure(figsize=(10, 6))
    plt.plot(radii, b0_vals, label=r'$\beta_0$ (Components)', color='blue', linewidth=2)
    plt.plot(radii, b1_vals, label=r'$\beta_1$ (Holes)', color='orange', linewidth=2, linestyle='--')
    
    # Heuristic markings
    # Find where B0 stabilizes to 1
    connected_indices = np.where(np.array(b0_vals) == 1)[0]
    if len(connected_indices) > 0:
        connect_radius = radii[connected_indices[0]]
        plt.axvline(connect_radius, color='green', linestyle=':', label=f'Connection Point (~{connect_radius:.2f})')
        
        # Highlight the "Plateau" region
        # Usually from Connection Point -> Where B1 starts dropping or becomes noise
        plt.axvspan(connect_radius, connect_radius + 1.5, color='green', alpha=0.1, label='Potential Plateau Region')

    plt.title(f"Topological Feature Evolution (k={fixed_k})")
    plt.xlabel("Eta (Radius/Distance)")
    plt.ylabel("Betti Number Count")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# --- Execution ---
if __name__ == "__main__":
    # 1. Plug in your Fixed K here (from previous step)
    FIXED_K_VALUE = 15  # <--- REPLACE THIS WITH YOUR MODE K
    
    # 2. Run Percentile Method (Statistical)
    # Assumes X_pca is loaded
    e25, e50, e75 = get_eta_candidates(X_pca, fixed_k=FIXED_K_VALUE, n_trials=20)
    
    # 3. Run Plateau Method (Visual)
    visualize_plateau(X_pca, fixed_k=FIXED_K_VALUE)